# 📚 Topic-Based Chunking Service using BERTopic
This notebook demonstrates how to build a topic-based chunking service using BERTopic.
It also handles the **-1 (outlier) topic** properly.

## 1. Install Dependencies

In [ ]:
!pip install bertopic sentence-transformers umap-learn hdbscan scikit-learn pandas

## 2. Imports

In [ ]:
import pandas as pd
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print('✅ All imports successful!')

## 3. Sample Clean Sentences
Replace this with your actual cleaned sentence list.

In [ ]:
# ✅ Replace this with your actual clean sentences
clean_sentences = [
    # Machine Learning
    "Machine learning is a subset of artificial intelligence.",
    "Neural networks are inspired by the human brain.",
    "Deep learning uses multiple layers to learn representations.",
    "Supervised learning requires labeled training data.",
    "Gradient descent is used to optimize model parameters.",
    "Overfitting occurs when a model memorizes training data.",

    # NLP
    "Natural language processing helps computers understand text.",
    "Transformers have revolutionized natural language processing.",
    "BERT is a pre-trained language model by Google.",
    "Tokenization splits text into smaller units called tokens.",
    "Word embeddings represent words as dense vectors.",

    # Climate
    "Climate change is one of the greatest global challenges.",
    "Rising sea levels threaten coastal communities worldwide.",
    "Carbon emissions are the primary driver of global warming.",
    "Renewable energy sources can reduce carbon footprints.",
    "Solar panels convert sunlight directly into electricity.",

    # Health
    "Regular exercise improves both mental and physical health.",
    "A balanced diet reduces the risk of chronic diseases.",
    "Sleep deprivation negatively affects cognitive performance.",
    "Vaccination programs have eradicated many deadly diseases.",

    # Outlier-like sentences (may get topic -1)
    "The quick brown fox jumps over the lazy dog.",
    "She sells seashells by the seashore.",
]

print(f'✅ Total sentences: {len(clean_sentences)}')

## 4. Configure BERTopic Components

In [ ]:
# --- Embedding Model ---
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')  # Fast & good quality

# --- UMAP: Dimensionality Reduction ---
# n_components: higher = less data loss but harder clustering
# n_neighbors: higher = more global structure preserved
umap_model = UMAP(
    n_neighbors=5,       # Smaller for small datasets; use 15 for large
    n_components=5,      # Reduced dimensions (increase to 10-15 to retain more info)
    min_dist=0.0,        # Tight clusters for better separation
    metric='cosine',
    random_state=42
)

# --- HDBSCAN: Clustering ---
# min_cluster_size: minimum sentences to form a topic
# min_samples: lower = more points assigned, fewer outliers (-1)
hdbscan_model = HDBSCAN(
    min_cluster_size=2,      # Adjust based on dataset size
    min_samples=1,           # Lower = fewer -1 outliers
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True     # Required for soft clustering (outlier reduction)
)

# --- Vectorizer ---
vectorizer_model = CountVectorizer(
    stop_words='english',
    min_df=1,
    ngram_range=(1, 2)       # Unigrams + bigrams for better topic keywords
)

# --- BERTopic Model ---
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    nr_topics='auto',        # Automatically find number of topics
    calculate_probabilities=True,  # Needed for outlier reduction
    verbose=True
)

print('✅ BERTopic model configured!')

## 5. Fit BERTopic and Get Topics

In [ ]:
# Fit the model
topics, probabilities = topic_model.fit_transform(clean_sentences)

print(f'\n✅ Topics assigned to {len(topics)} sentences')
print(f'📊 Unique topics found: {set(topics)}')
print(f'⚠️  Outlier (-1) count: {topics.count(-1)}')

## 6. View Topic Info

In [ ]:
topic_info = topic_model.get_topic_info()
print('\n📋 Topic Summary:')
print(topic_info.to_string(index=False))

## 7. ⚠️ Handle Topic -1 (Outliers)

**Topic -1 = sentences that HDBSCAN couldn't assign to any cluster.**  
We have 3 strategies to handle them:

In [ ]:
# ── Strategy 1: Reduce Outliers using Soft Clustering (RECOMMENDED) ──
# Uses probability scores to reassign -1 sentences to nearest topic

print('=== Strategy 1: Soft Clustering (BERTopic built-in) ===')

new_topics = topic_model.reduce_outliers(
    clean_sentences,
    topics,
    probabilities=probabilities,
    strategy='probabilities',  # Use probability threshold
    threshold=0.01             # Low threshold = assign most outliers to best topic
)

print(f'Before: {topics.count(-1)} outliers')
print(f'After:  {list(new_topics).count(-1)} outliers')
print(f'✅ Reduced outliers using soft clustering!')

In [ ]:
# ── Strategy 2: Semantic Similarity Fallback ──
# For remaining -1 topics, find the most similar topic by embedding similarity

from sklearn.metrics.pairwise import cosine_similarity

print('=== Strategy 2: Semantic Similarity Fallback ===')

def assign_outlier_by_similarity(sentences, topics, topic_model, embedding_model):
    """
    For sentences with topic -1, find the closest topic
    by computing cosine similarity between the sentence
    embedding and each topic's centroid embedding.
    """
    final_topics = list(topics)
    
    # Get valid topic IDs (excluding -1)
    valid_topics = [t for t in set(topics) if t != -1]
    
    if not valid_topics:
        print('⚠️ No valid topics found — all sentences are outliers!')
        return final_topics
    
    # Build topic centroid embeddings from their representative words
    topic_word_dict = {}
    for t in valid_topics:
        words = [word for word, _ in topic_model.get_topic(t)[:5]]
        topic_word_dict[t] = ' '.join(words)
    
    topic_ids = list(topic_word_dict.keys())
    topic_texts = [topic_word_dict[t] for t in topic_ids]
    topic_embeddings = embedding_model.encode(topic_texts)
    
    # Assign each -1 sentence to nearest topic
    outlier_indices = [i for i, t in enumerate(topics) if t == -1]
    
    if not outlier_indices:
        print('✅ No outliers to reassign!')
        return final_topics
    
    outlier_sentences = [sentences[i] for i in outlier_indices]
    outlier_embeddings = embedding_model.encode(outlier_sentences)
    
    similarities = cosine_similarity(outlier_embeddings, topic_embeddings)
    
    for idx, sim_row in zip(outlier_indices, similarities):
        best_topic_idx = np.argmax(sim_row)
        assigned_topic = topic_ids[best_topic_idx]
        confidence = sim_row[best_topic_idx]
        final_topics[idx] = assigned_topic
        print(f'  Sentence: "{sentences[idx][:50]}..."')
        print(f'  → Assigned to Topic {assigned_topic} (confidence: {confidence:.3f})\n')
    
    return final_topics


final_topics = assign_outlier_by_similarity(
    clean_sentences, new_topics, topic_model, embedding_model
)

print(f'✅ Final outlier count: {final_topics.count(-1)}')

In [ ]:
# ── Strategy 3: Keep -1 as a Separate "Misc" Chunk ──
# Sometimes outlier sentences are genuinely unrelated — keep them as their own chunk

print('=== Strategy 3: Treat -1 as Miscellaneous Chunk ===')
print('Sentences with topic -1 will be grouped into a "Miscellaneous" chunk.')
print('This is useful when outliers are truly off-topic and should not pollute other topics.\n')

# This is handled in the chunking function below via the misc_chunk strategy

## 8. 🏗️ Topic-Based Chunking Service

In [ ]:
class TopicChunkingService:
    """
    A service that takes clean sentences and groups them
    into topic-based chunks using BERTopic.
    
    Handles outlier topic -1 with configurable strategies:
      - 'similarity'  : Assign to nearest topic by cosine similarity
      - 'misc_chunk'  : Keep as a separate Miscellaneous chunk
      - 'drop'        : Discard outlier sentences
    """

    def __init__(self, topic_model, embedding_model, outlier_strategy='similarity'):
        """
        Args:
            topic_model      : Fitted BERTopic model
            embedding_model  : SentenceTransformer model
            outlier_strategy : How to handle topic -1
                               'similarity' | 'misc_chunk' | 'drop'
        """
        self.topic_model = topic_model
        self.embedding_model = embedding_model
        self.outlier_strategy = outlier_strategy

    def _get_topic_keywords(self, topic_id, top_n=5):
        """Get top keywords for a topic."""
        try:
            words = self.topic_model.get_topic(topic_id)
            return [w for w, _ in words[:top_n]] if words else [f'topic_{topic_id}']
        except:
            return [f'topic_{topic_id}']

    def _handle_outliers(self, sentences, topics):
        """Apply selected outlier handling strategy."""
        final_topics = list(topics)
        outlier_indices = [i for i, t in enumerate(final_topics) if t == -1]

        if not outlier_indices:
            print('✅ No outlier (-1) sentences found!')
            return final_topics

        print(f'\n⚠️  Found {len(outlier_indices)} outlier sentence(s) with topic -1')
        print(f'🔧 Applying strategy: "{self.outlier_strategy}"')

        if self.outlier_strategy == 'drop':
            # Mark as None to drop later
            for i in outlier_indices:
                final_topics[i] = None
            print(f'  → {len(outlier_indices)} sentences will be dropped.')

        elif self.outlier_strategy == 'misc_chunk':
            # Keep -1 as a special "misc" group
            print(f'  → {len(outlier_indices)} sentences kept as "Miscellaneous" chunk.')

        elif self.outlier_strategy == 'similarity':
            # Assign each outlier to most similar topic
            valid_topics = [t for t in set(final_topics) if t not in (-1, None)]
            if not valid_topics:
                print('  ⚠️ No valid topics available. Keeping as -1.')
                return final_topics

            # Get topic representative embeddings
            topic_texts = [' '.join(self._get_topic_keywords(t)) for t in valid_topics]
            topic_embeddings = self.embedding_model.encode(topic_texts)

            outlier_sents = [sentences[i] for i in outlier_indices]
            outlier_embeddings = self.embedding_model.encode(outlier_sents)

            sims = cosine_similarity(outlier_embeddings, topic_embeddings)

            for j, idx in enumerate(outlier_indices):
                best = np.argmax(sims[j])
                assigned = valid_topics[best]
                final_topics[idx] = assigned
                print(f'  → "{sentences[idx][:60]}" → Topic {assigned} '
                      f'(sim={sims[j][best]:.3f}, keywords: {self._get_topic_keywords(assigned)})')

        return final_topics

    def chunk(self, sentences, topics, probabilities=None):
        """
        Group sentences into topic-based chunks.

        Args:
            sentences    : List of clean sentences
            topics       : Topic IDs from BERTopic
            probabilities: Topic probabilities (for soft clustering)

        Returns:
            List of chunk dicts with topic_id, keywords, and sentences
        """
        # Step 1: First apply soft clustering if probs available
        if probabilities is not None:
            try:
                topics = self.topic_model.reduce_outliers(
                    sentences, topics,
                    probabilities=probabilities,
                    strategy='probabilities',
                    threshold=0.01
                )
                print(f'✅ Soft clustering done. Outliers after: {list(topics).count(-1)}')
            except Exception as e:
                print(f'⚠️ Soft clustering failed: {e}. Proceeding without it.')

        # Step 2: Handle remaining -1 outliers
        final_topics = self._handle_outliers(sentences, topics)

        # Step 3: Group sentences by topic
        chunks_dict = defaultdict(list)
        for sentence, topic in zip(sentences, final_topics):
            if topic is None:  # Dropped sentences
                continue
            chunks_dict[topic].append(sentence)

        # Step 4: Build output chunks
        chunks = []
        for topic_id, sents in sorted(chunks_dict.items()):
            if topic_id == -1:
                label = 'Miscellaneous'
                keywords = ['misc', 'other', 'unclassified']
            else:
                keywords = self._get_topic_keywords(topic_id)
                label = f'Topic_{topic_id}'

            chunks.append({
                'topic_id'   : topic_id,
                'label'      : label,
                'keywords'   : keywords,
                'sentences'  : sents,
                'chunk_text' : ' '.join(sents),   # Combined text for RAG/embedding
                'size'       : len(sents)
            })

        return chunks

    def display_chunks(self, chunks):
        """Pretty print the chunks."""
        print('\n' + '='*70)
        print('📦 TOPIC-BASED CHUNKS')
        print('='*70)
        for i, chunk in enumerate(chunks):
            print(f'\n🔷 Chunk {i+1}: {chunk["label"]}')
            print(f'   Topic ID  : {chunk["topic_id"]}')
            print(f'   Keywords  : {chunk["keywords"]}')
            print(f'   Size      : {chunk["size"]} sentences')
            print(f'   Sentences :')
            for s in chunk['sentences']:
                print(f'     - {s}')
        print('\n' + '='*70)


print('✅ TopicChunkingService class defined!')

## 9. Run the Chunking Service

In [ ]:
# ── Option A: Use 'similarity' strategy (recommended) ──
service = TopicChunkingService(
    topic_model=topic_model,
    embedding_model=embedding_model,
    outlier_strategy='similarity'  # or 'misc_chunk' or 'drop'
)

chunks = service.chunk(
    sentences=clean_sentences,
    topics=topics,
    probabilities=probabilities
)

service.display_chunks(chunks)

## 10. Export Chunks to DataFrame

In [ ]:
# Flatten chunks to a sentence-level dataframe
rows = []
for chunk in chunks:
    for sent in chunk['sentences']:
        rows.append({
            'sentence'   : sent,
            'topic_id'   : chunk['topic_id'],
            'label'      : chunk['label'],
            'keywords'   : ', '.join(chunk['keywords'])
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
# Chunk-level summary
chunk_summary = pd.DataFrame([{
    'topic_id' : c['topic_id'],
    'label'    : c['label'],
    'keywords' : ', '.join(c['keywords']),
    'size'     : c['size'],
    'chunk_text': c['chunk_text'][:80] + '...'
} for c in chunks])

print('\n📊 Chunk Summary:')
print(chunk_summary.to_string(index=False))

## 11. Visualize Topics (Optional)

In [ ]:
# Visualize topic distances (requires plotly)
try:
    fig = topic_model.visualize_topics()
    fig.show()
except Exception as e:
    print(f'Visualization skipped: {e}')

In [ ]:
# Visualize document-topic distribution
try:
    fig2 = topic_model.visualize_documents(clean_sentences)
    fig2.show()
except Exception as e:
    print(f'Document visualization skipped: {e}')

## 12. 🧠 Summary: How We Handle Topic -1

| Strategy | When to Use | Trade-off |
|---|---|---|
| **Soft Clustering** (BERTopic built-in) | First pass — always apply | May assign weakly related sentences |
| **Semantic Similarity** (`similarity`) | Remaining -1 after soft clustering | Best assignment by meaning |
| **Misc Chunk** (`misc_chunk`) | Outliers are genuinely off-topic | Preserves them but creates noise chunk |
| **Drop** (`drop`) | Outliers are truly irrelevant/noise | Information loss |

**Recommended pipeline:**
1. Use `calculate_probabilities=True` in BERTopic
2. Apply `reduce_outliers(..., strategy='probabilities')` first
3. For remaining -1, use `similarity` strategy
4. Only use `misc_chunk` or `drop` if outliers are truly noise